# High Noon — a quantitative teardown 🔬
### Real total-return tape · conditional forward returns · HAC inference · Wilson CIs

![Signal: None](https://img.shields.io/badge/Signal-None-c0392b?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![ATH a sell signal?: Busted](https://img.shields.io/badge/ATH_a_sell_signal%3F-Busted-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) — *same seven beats, every claim now carrying its standard error.* We split SPY days into *at-ATH* (within 1% of the running high) and *not-at-ATH*, compare forward 1/3/12-month returns with HAC *t*-stats and Wilson win-rate intervals, test the **difference**, and run the avoid-the-highs rule as a strategy.

> ⚠️ **Not investment advice.** SPY daily, total-return adjusted (`quantlab.data`, Yahoo); overlapping forward returns → HAC lags set to the horizon; cash earns 0% (conservative); 5 bps/switch; one-day execution lag. Sources in [`docs/references.md`](../docs/references.md), reproducible run in [`docs/results.md`](../docs/results.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # study root (high_noon/)
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5)
import numpy as np, pandas as pd
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from high_noon import data, strategy

AS_OF = "2026-06-12"
NEAR_PCT = 0.01
M = strategy.MONTH
frame = data.load_real("SPY", mode="total_return").loc[:AS_OF]
close = frame["close"]
flag = strategy.at_ath(close, NEAR_PCT)
print(f"SPY total return: {len(close):,} rows  {close.index[0].date()} -> {close.index[-1].date()}  fingerprint={data.fingerprint(frame)}")
print(f"days within 1% of the running all-time high: {flag.mean()*100:.1f}%")


SPY total return: 8,400 rows  1993-01-29 -> 2026-06-12  fingerprint=b8f3c9d7d95e
days within 1% of the running all-time high: 29.0%


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| **Signal** | `NONE` | at-minus-not forward-return difference = **-0.42 / -0.31 / +1.44 pts** at 1/3/12 mo, HAC *t* = **-1.34 / -0.36 / +0.38** — none clears \|t\|=2, and 12-mo is *positive*. |
| **Tradability** | `MIRAGE` | avoid-highs trails buy-and-hold by **1.7 pts/yr** (9.12% vs 10.82%), Sharpe -0.066, **0.6x** wealth, no drawdown relief. |
| **ATH a sell signal?** | `BUSTED` | 12-mo win-rate **84.6%** at-ATH vs **80.5%** not (Wilson intervals disjoint). |

> 💡 **In plain words:** the 'riskiest moment' is statistically an *ordinary* moment — and at a one-year horizon a slightly *above-average* one.

## 1 · The claim, steelmanned

- **H₁ (the folk rule):** forward returns conditional on being at/near an ATH are **significantly lower** than when not near a high.
- **H₀ (the null we expect):** they are indistinguishable (highs cluster in uptrends).
- **H₂ (the momentum inversion):** they might even be *higher* (George–Hwang 52-week-high).

## 2 · So what? — what rides on the answer

If H₁ held, a record would be a tradable fade and 'wait for the pullback' would be a free option. It doesn't — and the cost of believing it is the upside you skip.

## 3 · How we'd know — the protocol

Conditional forward-return means (HAC *t*, lag = horizon) · win-rates with Wilson 95% intervals · a HAC test of the **difference** (at − not) · the avoid-the-highs backtest vs buy-and-hold. Mirage trigger: the difference is not significantly negative at any horizon.

## 4 · The teardown

Conditional forward-return table — means, win-rates (Wilson 95%), and HAC *t* on each arm:

In [2]:
out = []
for h, lab in [(M,'1-month'),(3*M,'3-month'),(12*M,'12-month')]:
    s = strategy.conditional_forward_stats(close, h, NEAR_PCT)
    for arm, nm in [(s['at'],'at-ATH'),(s['not_at'],'not-ATH')]:
        out.append([lab, nm, arm['n'], arm['mean']*100, arm['win_rate']*100,
                    arm['win_lo']*100, arm['win_hi']*100, arm['hac_t']])
tbl = pd.DataFrame(out, columns=['horizon','arm','n','mean%','win%','win_lo','win_hi','HAC t'])
tbl.round(2)

,horizon,arm,n,mean%,win%,win_lo,win_hi,HAC t
0,1-month,at-ATH,2423,0.670,65.370,63.460,67.240,3.170
1,1-month,not-ATH,5956,1.090,65.880,64.670,67.080,4.720
2,3-month,at-ATH,2401,2.630,76.050,74.300,77.720,4.600
3,3-month,not-ATH,5936,2.940,69.910,68.730,71.070,4.420
4,12-month,at-ATH,2276,13.090,84.620,83.080,86.050,5.540
5,12-month,not-ATH,5872,11.650,80.470,79.430,81.460,3.980


**The test of the difference** — at-ATH minus not-at-ATH, with a HAC *t* (this is the line that decides Signal):

In [3]:
for h, lab in [(M,'1-month'),(3*M,'3-month'),(12*M,'12-month')]:
    s = strategy.conditional_forward_stats(close, h, NEAR_PCT)
    verdict = 'bearish' if s['diff_hac_t']<=-2 else 'NOT bearish'
    print(f"{lab:>9}: diff {s['diff_mean']*100:+5.2f} pts   HAC t {s['diff_hac_t']:+5.2f}   -> {verdict}")

  1-month: diff -0.42 pts   HAC t -1.34   -> NOT bearish
  3-month: diff -0.31 pts   HAC t -0.36   -> NOT bearish
 12-month: diff +1.44 pts   HAC t +0.38   -> NOT bearish


> 💡 **In plain words:** at no horizon is the all-time-high day a worse bet than the off-high day — the differences are noise, and the only one with a clear sign (12 months) is *positive*. The 'sell signal' isn't there.

**The rule as a strategy** — avoid the highs vs buy-and-hold, net of 5 bps/switch:

In [4]:
pos = strategy.avoid_highs_position(close, NEAR_PCT, lag=1)
timer = strategy.backtest(close, pos, cost_bps=5.0)
bh = strategy.buy_and_hold(close)
stt = pd.DataFrame({
  'CAGR %':[timer['cagr']*100, bh['cagr']*100],
  'Vol %':[timer['vol']*100, bh['vol']*100],
  'Sharpe':[timer['sharpe'], bh['sharpe']],
  'MaxDD %':[timer['max_dd']*100, bh['max_dd']*100],
  'TiM %':[timer['time_in_market']*100, bh['time_in_market']*100],
}, index=['Avoid-highs timer','Buy & hold'])
print(f"avoid-highs ends at {timer['final']/bh['final']:.2f}x buy-and-hold")
stt.round(2)

avoid-highs ends at 0.60x buy-and-hold


,CAGR %,Vol %,Sharpe,MaxDD %,TiM %
Avoid-highs timer,9.120,17.750,0.580,-54.660,71.000
Buy & hold,10.820,18.580,0.650,-55.190,100.000


> 💡 **In plain words:** the rule sheds 29% of the days — and they were disproportionately *good* days — so it loses on return and Sharpe while buying *no* drawdown protection.

## 5 · The verdict

Signal `NONE` (difference HAC *t* -1.34/-0.36/+0.38, none below −2), Tradability `MIRAGE` (−1.7 pts/yr, Sharpe -0.066, 0.6x wealth), ATH a sell signal? `BUSTED` (12-mo win 84.6% vs 80.5%). The folklore inverts the data.

## 6 · Could you trade it?

Capacity is a non-issue (SPY). The binding fact is economic: 'avoid the highs' is a **negative-edge** filter — it removes the trending days that carry the index's return while leaving the full drawdown intact. Adding taxes on 641 switches only widens the gap. There is nothing here to scale.

## 7 · Going further

- Sweep the band (0% = strict new highs, 5%, 10%) and the horizon — confirm the null is robust.
- Swap the all-time high for the **52-week high** and test the George–Hwang (2004) momentum claim directly.
- Block-bootstrap the 12-month difference for a CI on the (mildly positive) +1.4 pts.